In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import estraces
from tqdm import tqdm
import os

# S-BOX AES
sbox = (
    0x63, 0x7c, 0x77, 0x7b, 0xf2, 0x6b, 0x6f, 0xc5, 0x30, 0x01, 0x67, 0x2b, 0xfe, 0xd7, 0xab, 0x76,
    0xca, 0x82, 0xc9, 0x7d, 0xfa, 0x59, 0x47, 0xf0, 0xad, 0xd4, 0xa2, 0xaf, 0x9c, 0xa4, 0x72, 0xc0,
    0xb7, 0xfd, 0x93, 0x26, 0x36, 0x3f, 0xf7, 0xcc, 0x34, 0xa5, 0xe5, 0xf1, 0x71, 0xd8, 0x31, 0x15,
    0x04, 0xc7, 0x23, 0xc3, 0x18, 0x96, 0x05, 0x9a, 0x07, 0x12, 0x80, 0xe2, 0xeb, 0x27, 0xb2, 0x75,
    0x09, 0x83, 0x2c, 0x1a, 0x1b, 0x6e, 0x5a, 0xa0, 0x52, 0x3b, 0xd6, 0xb3, 0x29, 0xe3, 0x2f, 0x84,
    0x53, 0xd1, 0x00, 0xed, 0x20, 0xfc, 0xb1, 0x5b, 0x6a, 0xcb, 0xbe, 0x39, 0x4a, 0x4c, 0x58, 0xcf,
    0xd0, 0xef, 0xaa, 0xfb, 0x43, 0x4d, 0x33, 0x85, 0x45, 0xf9, 0x02, 0x7f, 0x50, 0x3c, 0x9f, 0xa8,
    0x51, 0xa3, 0x40, 0x8f, 0x92, 0x9d, 0x38, 0xf5, 0xbc, 0xb6, 0xda, 0x21, 0x10, 0xff, 0xf3, 0xd2,
    0xcd, 0x0c, 0x13, 0xec, 0x5f, 0x97, 0x44, 0x17, 0xc4, 0xa7, 0x7e, 0x3d, 0x64, 0x5d, 0x19, 0x73,
    0x60, 0x81, 0x4f, 0xdc, 0x22, 0x2a, 0x90, 0x88, 0x46, 0xee, 0xb8, 0x14, 0xde, 0x5e, 0x0b, 0xdb,
    0xe0, 0x32, 0x3a, 0x0a, 0x49, 0x06, 0x24, 0x5c, 0xc2, 0xd3, 0xac, 0x62, 0x91, 0x95, 0xe4, 0x79,
    0xe7, 0xc8, 0x37, 0x6d, 0x8d, 0xd5, 0x4e, 0xa9, 0x6c, 0x56, 0xf4, 0xea, 0x65, 0x7a, 0xae, 0x08,
    0xba, 0x78, 0x25, 0x2e, 0x1c, 0xa6, 0xb4, 0xc6, 0xe8, 0xdd, 0x74, 0x1f, 0x4b, 0xbd, 0x8b, 0x8a,
    0x70, 0x3e, 0xb5, 0x66, 0x48, 0x03, 0xf6, 0x0e, 0x61, 0x35, 0x57, 0xb9, 0x86, 0xc1, 0x1d, 0x9e,
    0xe1, 0xf8, 0x98, 0x11, 0x69, 0xd9, 0x8e, 0x94, 0x9b, 0x1e, 0x87, 0xe9, 0xce, 0x55, 0x28, 0xdf,
    0x8c, 0xa1, 0x89, 0x0d, 0xbf, 0xe6, 0x42, 0x68, 0x41, 0x99, 0x2d, 0x0f, 0xb0, 0x54, 0xbb, 0x16)


# 字节替换
def sub_bytes(pt, keyguess):
    return sbox[pt ^ keyguess]


# 创建用于存储图片的文件夹

num_of_traces = 8000
num_of_points = 50000


def compress_average(CH1):
    trace_num = CH1.shape[0]
    point_num = CH1.shape[1]
    trace = np.zeros((trace_num, point_num // 10), dtype=np.float64)
    for j in tqdm(range(point_num // 10), desc="取均值压缩"):
        trace[:, j] = np.mean(CH1[:, j * 10:j * 10 + 10], axis=1)
    return trace


ds2name = 'ds_atmega2560_aes_power.ets'
print(f"加载数据集2：{ds2name}")
ths_power = estraces.read_ths_from_ets_file(ds2name)
traces = ths_power.samples
pt = ths_power.metadatas['plaintext']
ckeys = ths_power.metadatas['key'][0]

traces = traces[:num_of_traces, :num_of_points]
traces = compress_average(traces[0:num_of_traces, :])
pt = pt[:num_of_traces, :]

trace_cnt = np.shape(traces)[0]
points_cnt = np.shape(traces)[1]
print("Number of traces: ", trace_cnt)
print(pt.shape, traces.shape)

# traces不转置！DPA：行=迹编号，列=采样点，【和原版CPA此处做修改】
# traces = traces.T

keys = []
kguesses = np.arange(256)

fig1, axes1 = plt.subplots(nrows=4, ncols=4, figsize=(12, 8))
fig2, axes2 = plt.subplots(nrows=4, ncols=4, figsize=(12, 8))

# 对每个字节进行DPA攻击
for pi in range(0, 16):
    print("\r第%d个字节" % (pi + 1), end="")
    pt_col = np.array([row[pi] for row in pt])  # 当前字节所有明文，shape(trace_cnt,)

    dpa_max_diff = np.zeros(256)  # 保存每个密钥猜测的最大差分绝对值

    for kg in range(256):
        # 计算每个迹对应的S‑box输出
        s_out = np.array([sub_bytes(p, kg) for p in pt_col])
        # DPA划分比特：取S输出最高位bit7
        bit_mask = (s_out >> 1) & 0x01

        group0 = traces[bit_mask == 0, :]
        group1 = traces[bit_mask == 1, :]

        mean0 = np.mean(group0, axis=0)
        mean1 = np.mean(group1, axis=0)
        diff = mean1 - mean0

        dpa_max_diff[kg] = np.max(np.abs(diff))

    recovered_byte = np.argmax(dpa_max_diff)
    keys.append(recovered_byte)

    axes1[pi // 4, pi % 4].plot(dpa_max_diff)
    axes2[pi // 4, pi % 4].plot(dpa_max_diff)

fig1.tight_layout()
fig2.tight_layout()
plt.show()


def display_key_analysis(recovered_keys, correct_keys):
    """
    对比并展示恢复的密钥与正确密钥的差异。
    :param recovered_keys: 攻击者恢复出的密钥列表 (bytes 或 list of ints)
    :param correct_keys: 真实的正确密钥列表 (bytes 或 list of ints)
    """
    print("\n" + "=" * 65)
    print(f"{'密钥字节分析 (Key Byte Analysis)':^60}")
    print("=" * 65)
    print(f"{'Idx':<6} | {'Recovered':<12} | {'Status':<8} | {'Correct':<12}")
    print("-" * 65)
    correct_count = 0
    for i in range(len(correct_keys)):
        rec_byte = recovered_keys[i] if i < len(recovered_keys) else None
        cor_byte = correct_keys[i]
        rec_str = f"0x{rec_byte:02x}" if rec_byte is not None else "None"
        cor_str = f"0x{cor_byte:02x}"
        if rec_byte == cor_byte:
            status = "✅ Match"
            correct_count += 1
        else:
            status = "❌ Error"
        print(f"{i:<6} | {rec_str:<12} | {status:<8} | {cor_str:<12}")
    print("-" * 65)
    total = len(correct_keys)
    accuracy = (correct_count / total) * 100 if total > 0 else 0
    print(f"\n📊 统计结果:")
    print(f"   总字节数: {total}")
    print(f"   正确匹配: {correct_count}")
    print(f"   攻击准确率: {accuracy:.1f}%")
    if accuracy == 100.0:
        print("   🎉 恭喜！所有子密钥均已成功恢复！")
    else:
        print("   ⚠️ 仍有部分密钥未恢复或错误。")
    print("=" * 65 + "\n")


print("The recovered Key is: ", )
for b in keys:
    print("%02x " % b, end="")

print("\nThe correct Key is: ", )
for b in ckeys:
    print("%02x " % b, end="")

display_key_analysis(keys, ckeys)
